In [ ]:
import os
import time
import copy
import torch
import gc 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

# ==========================================
# 1. Clase Dataset personalizada (Formato YOLO)
# ==========================================
class ContainerDamageDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.images_dir = os.path.join(root_dir, split, 'images')
        self.labels_dir = os.path.join(root_dir, split, 'labels')
        self.transform = transform
        
        if os.path.exists(self.images_dir):
            self.image_files = [f for f in os.listdir(self.images_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
        else:
            self.image_files = []

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        label_name = img_name.rsplit('.', 1)[0] + '.txt'
        label_path = os.path.join(self.labels_dir, label_name)

        label_id = 0 
        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            with open(label_path, 'r') as f:
                first_line = f.readline().strip()
                if first_line:
                    label_id = int(first_line.split()[0]) 

        if self.transform:
            image = self.transform(image)

        return image, label_id

# ==========================================
# 2. Transformaciones y Carga de Datos
# ==========================================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5), 
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

valid_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

DATASET_PATH = './dataset' 

train_dataset = ContainerDamageDataset(root_dir=DATASET_PATH, split='train', transform=train_transform)
valid_dataset = ContainerDamageDataset(root_dir=DATASET_PATH, split='valid', transform=valid_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False)

print(f"Imágenes de entrenamiento: {len(train_dataset)}")
print(f"Imágenes de validación: {len(valid_dataset)}")

# ==========================================
# 3. Configurar Dispositivo y Modelos
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

num_classes = 8 

print("Cargando ResNet50...")
resnet_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, num_classes)
resnet_model = resnet_model.to(device)

print("Cargando ViT_B_16...")
vit_model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
vit_model.heads.head = nn.Linear(vit_model.heads.head.in_features, num_classes)
vit_model = vit_model.to(device)

# ==========================================
# 4. Función de Entrenamiento
# ==========================================
def train_model(model, dataloaders, criterion, optimizer, num_epochs):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Época {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'valid']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'valid' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        print()

    time_elapsed = time.time() - since
    print(f'Entrenamiento completado en {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Mejor validación Acc: {best_acc:.4f}\n')

    model.load_state_dict(best_model_wts)
    return model

# ==========================================
# 5. Ejecutar Entrenamiento (15 Épocas)
# ==========================================

dataloaders_dict = {'train': train_loader, 'valid': valid_loader}
criterion = nn.CrossEntropyLoss()

print("--- Iniciando Entrenamiento ResNet50 (15 Épocas) ---")
optimizer_resnet = optim.Adam(resnet_model.parameters(), lr=1e-4)
resnet_model = train_model(resnet_model, dataloaders_dict, criterion, optimizer_resnet, num_epochs=15)

resnet_model = resnet_model.cpu() 
torch.cuda.empty_cache() 
gc.collect()

print("\n--- Iniciando Entrenamiento ViT (15 Épocas) ---")

optimizer_vit = optim.Adam(vit_model.parameters(), lr=1e-4)
vit_model = train_model(vit_model, dataloaders_dict, criterion, optimizer_vit, num_epochs=15)

print("\n" + "="*60)
print("🚢 CONCLUSIÓN TÉCNICA: PORT-EYE (RESNET50 vs ViT)")
print("="*60)
print("1. Viabilidad de Automatización (Eficacia):")
print("   Ambos modelos lograron aprender los patrones físicos de daño")
print("   (óxido, perforaciones, abolladuras) rondando el 60% de")
print("   precisión con apenas 770 imágenes. ResNet50 logró 59.66%")
print("   y ViT 59.09%, confirmando que la automatización es viable.\n")

print("2. Desempeño Operativo (Costo Computacional):")
print("   - ResNet50 procesó las 15 épocas en solo 4 minutos y 22 segundos.")
print("   - ViT requirió 12 minutos y 7 segundos (casi el triple de tiempo).\n")

print("3. Veredicto Logístico para el Puerto:")
print("   En el contexto de un terminal portuario real, donde el flujo")
print("   de camiones exige inferencia en tiempo real (milisegundos),")
print("   ResNet50 demostró ser superior: no solo fue ligeramente más")
print("   preciso, sino casi 3 veces más rápido computacionalmente.")
print("   Se consolida como la arquitectura definitiva para el check-gate.")
print("="*60 + "\n")

Imágenes de entrenamiento: 770
Imágenes de validación: 176
Usando dispositivo: cuda
Cargando ResNet50...
Cargando ViT_B_16...
--- Iniciando Entrenamiento ResNet50 (15 Épocas) ---
Época 1/15
----------
train Loss: 1.6353 Acc: 0.4688
valid Loss: 1.4819 Acc: 0.4886

Época 2/15
----------
train Loss: 1.1721 Acc: 0.5766
valid Loss: 1.4295 Acc: 0.4943

Época 3/15
----------
train Loss: 0.9330 Acc: 0.6623
valid Loss: 1.4390 Acc: 0.4830

Época 4/15
----------
train Loss: 0.7048 Acc: 0.7584
valid Loss: 1.4733 Acc: 0.5170

Época 5/15
----------
train Loss: 0.5096 Acc: 0.8390
valid Loss: 1.3828 Acc: 0.5455

Época 6/15
----------
train Loss: 0.3944 Acc: 0.8831
valid Loss: 1.5526 Acc: 0.5227

Época 7/15
----------
train Loss: 0.3098 Acc: 0.9260
valid Loss: 1.4185 Acc: 0.5966

Época 8/15
----------
train Loss: 0.2219 Acc: 0.9649
valid Loss: 1.3640 Acc: 0.5682

Época 9/15
----------
train Loss: 0.2020 Acc: 0.9494
valid Loss: 1.4326 Acc: 0.5398

Época 10/15
----------
train Loss: 0.1323 Acc: 0.9753
va